# **1.Thư viện**

In [1]:
%%capture
%pip install pyvi tabulate datasets pandas gdown -q

In [2]:
import pandas as pd
import gdown
import os
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pyvi import ViTokenizer
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

d:\Daniel_Nguyen\UIT_HK6\venv\Lib\site-packages\pyvi\ViTokenizer.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  model = pickle.load(fin)


# **2. Xử lý dữ liệu**

In [3]:
dataset_dir = 'dataset'
os.makedirs(dataset_dir, exist_ok=True)

# Danh sách ID từ link Google Drive gốc của nhóm tác giả
file_ids = {
    'train_sentences.txt': '1nzak5OkrheRV1ltOGCXkT671bmjODLhP',
    'train_sentiments.txt': '1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv',
    'train_topics.txt': '14MuDtwMnNOcr4z_8KdpxprjbwaQ7lJ_C',
    'val_sentences.txt': '1sMJSR3oRfPc3fe1gK-V3W5F24tov_517',
    'val_sentiments.txt': '1GiY1AOp41dLXIIkgES4422AuDwmbUseL',
    'val_topics.txt': '1DwLgDEaFWQe8mOd7EpF-xqMEbDLfdT-W',
    'test_sentences.txt': '1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n',
    'test_sentiments.txt': '1vkQS5gI0is4ACU58-AbWusnemw7KZNfO',
    'test_topics.txt': '1_ArMpDguVsbUGl-xSMkTF_p5KpZrmpSB'
}

print(f"Đang tải dữ liệu từ Google Drive vào thư mục '{dataset_dir}/'...")
for filename, file_id in file_ids.items():
    # 2. Tạo đường dẫn file nằm bên trong thư mục dataset
    filepath = os.path.join(dataset_dir, filename) 
    
    if not os.path.exists(filepath): # Chỉ tải nếu file chưa tồn tại
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, filepath, quiet=True)

print("Đã tải xong! Bắt đầu ráp dữ liệu...")

# Hàm đọc và ráp 3 file txt thành 1 DataFrame
def create_dataframe(sentences_name, sentiments_name, topics_name):
    sentences_path = os.path.join(dataset_dir, sentences_name)
    sentiments_path = os.path.join(dataset_dir, sentiments_name)
    topics_path = os.path.join(dataset_dir, topics_name)

    with open(sentences_path, 'r', encoding='utf-8') as f_sent:
        sentences = [line.strip() for line in f_sent]
    with open(sentiments_path, 'r', encoding='utf-8') as f_senti:
        sentiments = [int(line.strip()) for line in f_senti]
    with open(topics_path, 'r', encoding='utf-8') as f_top:
        topics = [int(line.strip()) for line in f_top]
        
    return pd.DataFrame({
        'sentence': sentences,
        'sentiment': sentiments,
        'topic': topics
    })

# Khởi tạo 3 tập dữ liệu
df_train = create_dataframe('train_sentences.txt', 'train_sentiments.txt', 'train_topics.txt')
df_val = create_dataframe('val_sentences.txt', 'val_sentiments.txt', 'val_topics.txt')
df_test = create_dataframe('test_sentences.txt', 'test_sentiments.txt', 'test_topics.txt')

print("Hoàn tất ráp dữ liệu!")
print(f"Kích thước tập Train: {df_train.shape}")
print(f"Kích thước tập Val: {df_val.shape}")
print(f"Kích thước tập Test: {df_test.shape}")

# Hiển thị thử 5 dòng đầu
df_train.head()

Đang tải dữ liệu từ Google Drive vào thư mục 'dataset/'...
Đã tải xong! Bắt đầu ráp dữ liệu...
Hoàn tất ráp dữ liệu!
Kích thước tập Train: (11426, 3)
Kích thước tập Val: (1583, 3)
Kích thước tập Test: (3166, 3)


,sentence,sentiment,topic
0,slide giáo trình đầy đủ .,2,1
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,0
2,đi học đầy đủ full điểm chuyên cần .,0,1
3,chưa áp dụng công nghệ thông tin và các thiết ...,0,0
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",2,0


In [4]:
def tokenize_text(text):
    # Kiểm tra nếu text không phải chuỗi (tránh lỗi dữ liệu trống)
    if not isinstance(text, str):
        return ""
    return ViTokenizer.tokenize(text)

print("⏳ Đang tiến hành tách từ tiếng Việt và trích xuất biến...")

# 2. Tách biến X (văn bản) và y (nhãn cảm xúc) cho tập TRAIN
# Vừa tách từ vừa gán biến để đảm bảo các "anh bạn" này được defined
X_train_text = df_train['sentence'].apply(tokenize_text)
y_train = df_train['sentiment']

# 3. Tách biến cho tập VALIDATION
X_val_text = df_val['sentence'].apply(tokenize_text)
y_val = df_val['sentiment']

# 4. Tách biến cho tập TEST
X_test_text = df_test['sentence'].apply(tokenize_text)
y_test = df_test['sentiment']

print("✅ Khai báo X_train_text, y_train, ... thành công!")
print(f"Ví dụ câu sau khi tách từ: {X_train_text.iloc[0]}")

⏳ Đang tiến hành tách từ tiếng Việt và trích xuất biến...
✅ Khai báo X_train_text, y_train, ... thành công!
Ví dụ câu sau khi tách từ: slide giáo_trình đầy_đủ .


In [5]:
common_params = {'ngram_range': (2, 2), 'min_df': 2, 'max_df': 0.90}
#Tác giả chỉ sử dụng bigrams

vectorizers = {
    "CountVectorizer": CountVectorizer(**common_params),
    "TfidfVectorizer": TfidfVectorizer(**common_params)
}

count_vec = vectorizers["CountVectorizer"]
tfidf_vec = vectorizers["TfidfVectorizer"]

In [6]:
X_train_count = count_vec.fit_transform(X_train_text)
X_train_tfidf = tfidf_vec.fit_transform(X_train_text)

X_val_count = count_vec.transform(X_val_text)
X_test_count = count_vec.transform(X_test_text)

X_val_tfidf = tfidf_vec.transform(X_val_text)
X_test_tfidf = tfidf_vec.transform(X_test_text)

Trong bài báo, nhóm tác giả đã dùng những đặc trưng đơn giản như *n-grams* (n đại diện cho số từ trong một cụm), cụ thể là *bigrams* cho việc truy xuất các đặc trưng.

Vì trong tiếng Việt (đặc biệt là nhận xét), cảm xúc thường nằm ở các cụm từ chéo nhau, điển hình là câu phủ định.
- Nếu dùng từ đơn: Chữ "*không*" và chữ "*tốt*" tách rời nhau, máy sẽ bối rối không biết đây là khen hay chê.
- Nếu dùng bigrams: Máy sẽ bắt được nguyên cụm "*không tốt*". Lúc này, ý nghĩa phủ định được giữ nguyên vẹn.

Cho nên:
- Đối với tập train, ta dùng hàm ```fit_transform``` để mô hình đọc tập dữ liệu. Từ đó hình thành bộ từ điển của riêng nó. Sau đó biến đổi các câu thành một ma trận số.
- Đối với tập validation và tập test, ta hình hàm ```transform```. Nếu mô hình gặp các từ không biết ở tập train, nó sẽ phớt lờ và bỏ qua từ. (Vì không được phép học trên 2 tập này á)

Tuy nhiên, họ không đề cập cụ thể mô hình nào trong việc xử lý dữ liệu, em dùng 2 mô hình đơn giản trên để thử kiểm chứng độc lập để so sánh.
- CountVectorizer: Trọng tâm của nó là tần suất, *chữ nào xuất hiện nhiều lần trong 1 câu thì chữ đó quan trọng*.
- TfidfVectorizer: Trọng tâm của nó là độ hiếm, *chữ nào xuất hiện nhiều trong 1 câu, nhưng ít xuất hiện ở các câu khác thì mới thực sự là từ khóa quan trọng*.

# **3. Mô hình học dữ liệu**

Trong bài báo, nhóm tác giả mang cái ma trận bigrams khổng lồ đó đi thi đấu trên 2 mô hình cơ sở (Baseline) là:
  1. Naive Bayes (Sử dụng framework Datumbox)
  2. Maximum Entropy - MaxEnt (Sử dụng Stanford Classifier)

Trong bài báo đưa ra kết luận là mô hình Maximum Entropy tốt hơn.

In [7]:
models = {
    "Maximum Entropy (Logistic Regression)": LogisticRegression(
        solver='lbfgs', max_iter=2000, class_weight='balanced', random_state=42
    ),
    "Naive Bayes (MultinomialNB)": MultinomialNB() 
}

In [8]:
from sklearn.pipeline import Pipeline
import copy

overall_results = []
detailed_reports = {}

target_names = ['Tiêu cực', 'Trung lập', 'Tích cực'] 

# Chạy vòng lặp để ghép cặp từng Vectorizer với từng Model
for vec_name, vec in vectorizers.items():
    for model_name, base_model in models.items():
        combo_name = f"{vec_name} + {model_name}" 
        
        # Tạo bản sao độc lập (Deepcopy) để tránh ghi đè model và vectorizer. Đây là Best Practice
        current_vec = copy.deepcopy(vec)
        current_model = copy.deepcopy(base_model)
        
        # Tạo Pipeline đóng gói Vectorizer và Classifier
        pipeline = Pipeline([
            ('vectorizer', current_vec),
            ('classifier', current_model)
        ])
        
        # Vừa biến đổi văn bản (fit_transform) vừa huấn luyện (fit) trên tập TRAIN
        pipeline.fit(X_train_text, y_train)
        
        # Dự đoán thẳng trên câu văn của tập TEST
        # (Pipeline tự động transform X_test_text sang số rồi mới đưa vào classifier dự đoán)
        y_test_pred = pipeline.predict(X_test_text)
        
        #Đánh giá
        acc = accuracy_score(y_test, y_test_pred)
        f1_macro = f1_score(y_test, y_test_pred, average='macro')
        
        overall_results.append({
            "Mô hình (Model)": combo_name,
            "Accuracy (%)": round(acc * 100, 2),
            "Macro F1 (%)": round(f1_macro * 100, 2)
        })
        
        report_dict = classification_report(y_test, y_test_pred, target_names=target_names, output_dict=True)
        detailed_reports[combo_name] = report_dict

# **4. Kết quả với các cách xử lý và học dữ liệu**

In [9]:
print("="*60)
print("BẢNG 1: TỔNG HỢP HIỆU SUẤT CÁC MÔ HÌNH (OVERALL PERFORMANCE)")
print("="*60)
df_overall = pd.DataFrame(overall_results).sort_values(by="Accuracy (%)", ascending=False)
print(df_overall.to_markdown(index=False))
print("\n")

BẢNG 1: TỔNG HỢP HIỆU SUẤT CÁC MÔ HÌNH (OVERALL PERFORMANCE)
| Mô hình (Model)                                         |   Accuracy (%) |   Macro F1 (%) |
|:--------------------------------------------------------|---------------:|---------------:|
| CountVectorizer + Naive Bayes (MultinomialNB)           |          86.17 |          62.47 |
| TfidfVectorizer + Naive Bayes (MultinomialNB)           |          85.88 |          58.79 |
| TfidfVectorizer + Maximum Entropy (Logistic Regression) |          81.87 |          65.22 |
| CountVectorizer + Maximum Entropy (Logistic Regression) |          80.26 |          66.75 |




In [10]:
best_model_name = df_overall.iloc[0]["Mô hình (Model)"]
best_report = detailed_reports[best_model_name]

In [11]:
class_metrics = []
for label in target_names:
    class_metrics.append({
        "Nhãn (Class)": label,
        "Precision (%)": round(best_report[label]['precision'] * 100, 2),
        "Recall (%)": round(best_report[label]['recall'] * 100, 2),
        "F1-Score (%)": round(best_report[label]['f1-score'] * 100, 2),
        "Số lượng (Support)": int(best_report[label]['support'])
    })

In [12]:
print("="*70)
print(f"BẢNG 2: KẾT QUẢ CHI TIẾT TỪNG NHÃN CỦA MÔ HÌNH TỐT NHẤT")
print(f"(Mô hình: {best_model_name})")
print("="*70)
df_detailed = pd.DataFrame(class_metrics)
print(df_detailed.to_markdown(index=False))

BẢNG 2: KẾT QUẢ CHI TIẾT TỪNG NHÃN CỦA MÔ HÌNH TỐT NHẤT
(Mô hình: CountVectorizer + Naive Bayes (MultinomialNB))
| Nhãn (Class)   |   Precision (%) |   Recall (%) |   F1-Score (%) |   Số lượng (Support) |
|:---------------|----------------:|-------------:|---------------:|---------------------:|
| Tiêu cực       |           85.31 |        90.7  |          87.93 |                 1409 |
| Trung lập      |           40    |         5.99 |          10.42 |                  167 |
| Tích cực       |           87.64 |        90.57 |          89.08 |                 1590 |


# **GIẢI THÍCH**

Trong bài báo *A Maximum Entropy Approach to Natural Language Processing* *https://aclanthology.org/J96-1002.pdf*, nhóm tác giả đã đề cập rằng: *"model all that is known and assume nothing about that which is unknown. In other words, given a collection of facts, choose a model which is consistent with all the facts, but otherwise as uniform as possible."* (**mô hình hóa tất cả những gì đã biết và không giả định bất cứ điều gì về những thứ chưa biết. Nói cách khác, khi có một tập hợp các sự kiện, hãy chọn một mô hình khớp với tất cả các sự kiện đó, nhưng ở các khía cạnh khác thì càng đồng đều càng tốt (đồng đều ở đây trong toán học chính là Entropy đạt cực đại).**)

Để dễ hiểu, giả sử ta có một viên xúc xắc 6 mặt. Bài toán yêu cầu ta đoán xác suất rơi vào từng mặt.
- Khi không có sự kiện nào (Không có thông tin): Ta sẽ chia đều xác suất là $1/6$ cho mỗi mặt. Việc chia đều này chính là trạng thái Đồng nhất nhất (Entropy cực đại). Ta *không giả định gì thêm* vì viên xúc xắc trông rất bình thường.
- Khi có sự kiện (Ràng buộc thông tin): Bây giờ bài toán tiết lộ một sự kiện: *"Viên xúc xắc này bị lỗi, mặt số 6 bị làm nặng hơn, xác suất ra mặt số 6 là 30%"*.

**Lúc này, chúng ta sẽ đoán thế nào?**

Nguyên lý Maximum Entropy bảo rằng: Hãy ghi nhận sự kiện mặt 6 là $30\%$, nhưng đối với 5 mặt còn lại (những thứ chúng ta chưa biết), hãy tiếp tục chia đều xác suất cho chúng (mỗi mặt chiếm $14\%$). Chúng ta không được tự tiện cho rằng mặt số 1 dễ ra hơn mặt số 2, vì chúng ta không có bằng chứng.

$\rightarrow$ Nguyên lý Maximum Entropy sẽ ép mô hình phải học thuộc những gì có thật trong dữ liệu (sự kiện), phần còn lại thì san bằng (đồng nhất).

Nói cách khác: nguyên lý Maximum Entropy ưu tiên việc xây dựng một mô hình ít đưa ra các giả định chủ quan nhất về những dữ liệu chưa biết (tức là tối đa hóa Entropy). Khi giải bài toán tối ưu hóa này dưới các ràng buộc tuyến tính từ tập dữ liệu huấn luyện, nghiệm toán học thu được chính là hàm Softmax. Họ sử dụng một công cụ giải tích gọi là *Nhân tử Lagrange*. Bất kể dữ liệu là gì, khi giải phương trình tối đa hóa Entropy đó, đáp án (nghiệm) luôn luôn có dạng một hàm số mũ:
$$
P(y|x) = \frac{\exp(W \cdot X)}{\sum \exp(W \cdot X)}
$$

*Trong đó:*
* $P(y|x)$: Xác suất dữ liệu $x$ thuộc về nhãn $y$.
* $W$: Ma trận trọng số (Weights) mà mô hình học được.
* $X$: Vector đặc trưng đầu vào (ma trận số sau khi Vector hóa).
* $\exp$: Hàm số mũ cơ số $e$ ($e \approx 2.718$).

Tức là **Hàm Softmax chính là KẾT QUẢ (nghiệm toán học) của quá trình đi tìm Entropy cực đại.**


Trong khi đó, bản chất của Logistic Regression đa lớp là đi tìm một đường ranh giới phân chia các lớp (Tiêu cực, Trung lập, Tích cực). Để biến khoảng cách từ điểm dữ liệu đến đường ranh giới đó thành xác suất $0\% \rightarrow 100\%$, Logistic Regression cũng sử dụng một hàm toán học tên là hàm Softmax. Mà vì hàm Softmax lại chính là hàm kích hoạt cốt lõi được sử dụng ở lớp đầu ra của mô hình Multinomial Logistic Regression để tính toán xác suất cho từng nhãn dự đoán.

**Cho nên, phương trình toán học nền tảng của Maximum Entropy và Multinomial Logistic Regression là hoàn toàn giống nhau.**